# 🐱 BRAT Classification
### EfficientNet-B0 Fine-Tuning with 5-Fold Stratified Cross-Validation

**Goal:** Binary image classification — does the cat image show a loaf position (paws tucked under body) or not?

---
## Dependencies

In [ ]:
!pip install torch torchvision scikit-learn matplotlib pillow

: 

---
## 1. Imports & Configuration

In [ ]:
import copy
import json
import time
import warnings
from pathlib import Path
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, precision_score,
    recall_score, roc_auc_score, roc_curve, auc,
)
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

warnings.filterwarnings("ignore", category=UserWarning)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"MPS      : {torch.backends.mps.is_available()}")

In [ ]:
# Labels
CLASS_TO_IDX   = {"cats": 0, "loafs": 1}
IDX_TO_CLASS   = {0: "cats", 1: "loafs"}
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

# Hyperparameters
DATA_DIR      = "images"
RESULTS_DIR   = "results"
IMG_SIZE      = 224
FINETUNE_MODE = "partial"
DROPOUT       = 0.4
EPOCHS        = 20
K_FOLDS       = 5
BATCH_SIZE    = 32
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 4
SEED          = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

def get_device() -> torch.device:
    if torch.cuda.is_available():        return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print(f"Device   : {DEVICE}")

---
## 2. Transforms

In [ ]:
def get_transforms(split: str = "train", img_size: int = IMG_SIZE) -> transforms.Compose:
    if split == "train":
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    else:  # val / test
        return transforms.Compose([
            transforms.Resize(int(img_size * 256 / 224)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

print("Train transforms:")
print(get_transforms("train"))
print("\nVal transforms:")
print(get_transforms("val"))

---
## 3. Dataset & DataLoader

In [ ]:
class CatLoafDataset(Dataset):
    def __init__(self, root: str, transform=None):
        self.root      = Path(root)
        self.transform = transform
        self.samples: List[Tuple[Path, int]] = []

        for cls_name, label in CLASS_TO_IDX.items():
            cls_dir = self.root / cls_name
            if not cls_dir.exists():
                raise FileNotFoundError(
                    f"Missing directory: {cls_dir}\n"
                    f"Expected 'loafs/' and 'cats/' inside {root}"
                )
            for fpath in sorted(cls_dir.iterdir()):
                if fpath.suffix.lower() in IMG_EXTENSIONS:
                    self.samples.append((fpath, label))

        if not self.samples:
            raise RuntimeError(f"No images found under {self.root}")

        n_loafs = sum(l for _, l in self.samples)
        n_cats  = len(self.samples) - n_loafs
        print(f"[Dataset] {len(self.samples)} total | loafs: {n_loafs} | cats: {n_cats}")

    @property
    def labels(self) -> List[int]:
        return [lbl for _, lbl in self.samples]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except (UnidentifiedImageError, OSError):
            img = Image.new("RGB", (224, 224), color=0)
        if self.transform:
            img = self.transform(img)
        return img, label

    def class_sample_weights(self) -> torch.Tensor:
        """Per-sample weights for WeightedRandomSampler (handles class imbalance)."""
        labels = self.labels
        n      = len(labels)
        n_pos  = max(sum(labels), 1)
        n_neg  = max(n - n_pos, 1)
        w      = {0: n / (2 * n_neg), 1: n / (2 * n_pos)}
        return torch.tensor([w[l] for l in labels], dtype=torch.float)

In [ ]:
def make_loader(
    base_dataset: CatLoafDataset,
    indices: List[int],
    split: str,
) -> DataLoader:
    """Build a DataLoader for a subset of indices with the correct transforms."""
    ds     = CatLoafDataset(base_dataset.root, transform=get_transforms(split))
    subset = Subset(ds, indices)

    if split == "train":
        weights = base_dataset.class_sample_weights()[indices]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        return DataLoader(subset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
    else:
        return DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

base_dataset = CatLoafDataset(DATA_DIR)

In [ ]:
DENORM_MEAN = np.array(IMAGENET_MEAN)
DENORM_STD  = np.array(IMAGENET_STD)

def denorm(tensor):
    img = tensor.permute(1, 2, 0).numpy()
    return np.clip(img * DENORM_STD + DENORM_MEAN, 0, 1)

preview_ds = CatLoafDataset(DATA_DIR, transform=get_transforms("val"))
samples_by_class = {0: [], 1: []}
for img, lbl in preview_ds:
    if len(samples_by_class[lbl]) < 5:
        samples_by_class[lbl].append(img)
    if all(len(v) == 5 for v in samples_by_class.values()):
        break

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for row, (cls_idx, cls_name) in enumerate([(0, "cats (not loaf)"), (1, "loafs")]):
    for col, img in enumerate(samples_by_class[cls_idx]):
        axes[row][col].imshow(denorm(img))
        axes[row][col].axis("off")
    axes[row][0].set_ylabel(cls_name, fontsize=12, fontweight="bold")

fig.suptitle("Sample Images by Class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Model — EfficientNet-B0

EfficientNet-B0 scales network **width, depth, and resolution together** using a fixed compound coefficient — like upgrading a car's engine, brakes, and suspension in proportion rather than just one part.

**Partial fine-tuning:** `features.0–6` stay frozen (general ImageNet features). Only `features.7`, `features.8`, and the new binary head are trained.

In [ ]:
class EfficientNetLoaf(nn.Module):
    def __init__(self, finetune_mode: str = FINETUNE_MODE, dropout: float = DROPOUT):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

        in_features = backbone.classifier[1].in_features 
        backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout, inplace=True),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout / 2),
            nn.Linear(256, 2),
        )

        self.backbone      = backbone
        self.finetune_mode = finetune_mode
        self._apply_finetune_mode(finetune_mode)

    def _apply_finetune_mode(self, mode: str) -> None:
        if mode == "head_only":
            for name, p in self.backbone.named_parameters():
                p.requires_grad = "classifier" in name
        elif mode == "partial":
            for p in self.backbone.parameters():
                p.requires_grad = False
            for name, p in self.backbone.named_parameters():
                if any(k in name for k in ("features.7", "features.8", "classifier")):
                    p.requires_grad = True
        elif mode == "full":
            for p in self.backbone.parameters():
                p.requires_grad = True
        else:
            raise ValueError(f"Unknown finetune_mode: '{mode}'")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

    def n_trainable(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def n_total(self) -> int:
        return sum(p.numel() for p in self.parameters())

_m = EfficientNetLoaf()
print(f"Trainable params : {_m.n_trainable():,}")
print(f"Total params     : {_m.n_total():,}")
print(f"Frozen params    : {_m.n_total() - _m.n_trainable():,}")
del _m

---
## 5. Training Utilities

In [ ]:
def compute_metrics(y_true, y_pred, y_prob) -> dict:
    """Compute accuracy, F1, precision, recall, and AUC-ROC."""
    return {
        "accuracy":  round(float(accuracy_score(y_true, y_pred)), 4),
        "f1":        round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "recall":    round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "auc_roc":   round(float(roc_auc_score(y_true, y_prob)), 4),
    }


def run_epoch(model, loader, criterion, optimizer, device, training: bool):
    """
    Run one full pass (train or val) over the data.
    Returns (avg_loss, metrics_dict).
    """
    model.train() if training else model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss   = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            total_loss  += loss.item() * imgs.size(0)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    avg_loss = total_loss / max(len(loader.dataset), 1)
    return avg_loss, compute_metrics(all_labels, all_preds, all_probs)

---
## 6. 5-Fold Cross-Validation

In [ ]:
def train_fold(
    fold:        int,
    train_idx,
    val_idx,
    base_dataset: CatLoafDataset,
    device:      torch.device,
    results_dir: Path,
) -> dict:
    print(f"\n{'='*62}")
    print(f"  FOLD {fold+1}/{K_FOLDS}  |  train={len(train_idx)}  val={len(val_idx)}")
    print(f"{'='*62}")

    train_loader = make_loader(base_dataset, list(train_idx), "train")
    val_loader   = make_loader(base_dataset, list(val_idx),   "val")

    model     = EfficientNetLoaf().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_val_f1, best_state, history = 0.0, None, []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_met = run_epoch(model, train_loader, criterion, optimizer, device, training=True)
        vl_loss, vl_met = run_epoch(model, val_loader,   criterion, None,      device, training=False)
        scheduler.step()

        print(
            f"  Ep {epoch:>3}/{EPOCHS} | "
            f"tr_loss={tr_loss:.4f} acc={tr_met['accuracy']:.4f} | "
            f"vl_loss={vl_loss:.4f} acc={vl_met['accuracy']:.4f} "
            f"f1={vl_met['f1']:.4f} auc={vl_met['auc_roc']:.4f} | "
            f"{time.time()-t0:.1f}s"
        )

        history.append({
            "epoch":      epoch,
            "train_loss": tr_loss, **{f"train_{k}": v for k, v in tr_met.items()},
            "val_loss":   vl_loss, **{f"val_{k}":   v for k, v in vl_met.items()},
        })

        if vl_met["f1"] > best_val_f1:
            best_val_f1 = vl_met["f1"]
            best_state  = copy.deepcopy(model.state_dict())
            print(f"    ✓ Best val F1: {best_val_f1:.4f}")

    ckpt = results_dir / f"fold_{fold+1}_best.pt"
    torch.save(best_state, ckpt)
    print(f"  Saved: {ckpt}")
    return {"fold": fold + 1, "best_val_f1": best_val_f1, "history": history}

In [ ]:
def run_cross_validation() -> dict:
    results_dir = Path(RESULTS_DIR)
    results_dir.mkdir(parents=True, exist_ok=True)

    print(f"[Config] device={DEVICE} | folds={K_FOLDS} | epochs={EPOCHS} | "
          f"lr={LR} | batch={BATCH_SIZE} | mode={FINETUNE_MODE}")

    labels  = np.array(base_dataset.labels)
    indices = np.arange(len(base_dataset))
    skf     = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    fold_results = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        result = train_fold(fold, train_idx, val_idx, base_dataset, DEVICE, results_dir)
        fold_results.append(result)

    f1s = [r["best_val_f1"] for r in fold_results]
    summary = {
        "model":         "EfficientNet-B0",
        "finetune_mode": FINETUNE_MODE,
        "k_folds":       K_FOLDS,
        "epochs":        EPOCHS,
        "mean_val_f1":   round(float(np.mean(f1s)), 4),
        "std_val_f1":    round(float(np.std(f1s)),  4),
        "per_fold_f1":   [round(s, 4) for s in f1s],
        "folds":         fold_results,
    }

    with open(Path(RESULTS_DIR) / "cv_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n{'='*62}")
    print(f"  5-FOLD CV COMPLETE")
    print(f"  Per-fold F1 : {summary['per_fold_f1']}")
    print(f"  Mean F1     : {summary['mean_val_f1']:.4f} ± {summary['std_val_f1']:.4f}")
    print(f"{'='*62}")
    return summary


# Training
cv_summary = run_cross_validation()

---
## 7. Evaluation & Plots

In [ ]:
def load_best_model(ckpt_path: Path) -> EfficientNetLoaf:
    model = EfficientNetLoaf(finetune_mode="full")
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return model.to(DEVICE).eval()


@torch.no_grad()
def predict_loader(model, loader):
    all_labels, all_preds, all_probs = [], [], []
    for imgs, labels in loader:
        logits = model(imgs.to(DEVICE))
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [ ]:
results_dir  = Path(RESULTS_DIR)
labels_arr   = np.array(base_dataset.labels)
indices_arr  = np.arange(len(base_dataset))
skf_eval     = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

all_labels, all_preds, all_probs = [], [], []
fold_metrics = []

for fold, (_, val_idx) in enumerate(skf_eval.split(indices_arr, labels_arr)):
    ckpt = results_dir / f"fold_{fold+1}_best.pt"
    if not ckpt.exists():
        print(f"[WARN] {ckpt} not found — skipping")
        continue

    print(f"[Fold {fold+1}] Evaluating {ckpt.name}")
    model  = load_best_model(ckpt)
    val_ds = CatLoafDataset(DATA_DIR, transform=get_transforms("val"))
    loader = DataLoader(Subset(val_ds, list(val_idx)), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    y_true, y_pred, y_prob = predict_loader(model, loader)
    all_labels.extend(y_true); all_preds.extend(y_pred); all_probs.extend(y_prob)

    fold_metrics.append({
        "fold":     fold + 1,
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "f1":       round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "auc_roc":  round(float(roc_auc_score(y_true, y_prob)), 4),
    })

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)

print(f"\n{'='*62}")
print("  AGGREGATE RESULTS — EfficientNet-B0 (5-fold CV)")
print(f"{'='*62}")
print(classification_report(all_labels, all_preds, target_names=["cats", "loafs"]))

f1s  = [m["f1"]      for m in fold_metrics]
aucs = [m["auc_roc"] for m in fold_metrics]
print(f"Per-fold F1  : {[m['f1'] for m in fold_metrics]}")
print(f"Mean F1      : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"Mean AUC-ROC : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

In [ ]:
# Confusion matrix
cm   = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["cats", "loafs"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Aggregated Confusion Matrix\n(all 5 folds)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(results_dir / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc     = auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, lw=2, color="#1a1a2e", label=f"EfficientNet-B0 (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (aggregated over 5 folds)", fontsize=11, fontweight="bold")
ax.legend(loc="lower right")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(results_dir / "roc_curve.png", dpi=150)
plt.show()

In [ ]:
# Training curves (val loss + val F1 per fold)
with open(results_dir / "cv_summary.json") as f:
    summary = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = plt.cm.tab10.colors

for fd in summary["folds"]:
    hist   = fd["history"]
    epochs = [h["epoch"]    for h in hist]
    v_loss = [h["val_loss"] for h in hist]
    v_f1   = [h["val_f1"]   for h in hist]
    c = colors[(fd["fold"] - 1) % len(colors)]
    axes[0].plot(epochs, v_loss, color=c, label=f"Fold {fd['fold']}")
    axes[1].plot(epochs, v_f1,   color=c, label=f"Fold {fd['fold']}")

for ax, ylabel, title in zip(
    axes,
    ["Validation Loss", "Validation F1"],
    ["Val Loss per Fold", "Val F1 per Fold"],
):
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

fig.suptitle("EfficientNet-B0 — 5-Fold Training Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(results_dir / "training_curves.png", dpi=150)
plt.show()